# Advanced Lapse Prediction Using XGBoost
### Probabilistic Model for Non-Life Insurance Flood Risk

**Authors:** Sebastian Carino, Ma. Loisa Celorico, Cyrill Agustin, John Navarro, and Justino Rotairo III  
**Notebook:** 07B — Advanced Cancellation and Non-Renewal Model

## 1. Research Objective and Method
This notebook evaluates whether XGBoost improves policy-year lapse probability estimates relative to the naive, logistic regression, and Random Forest benchmarks. The analysis uses a prespecified set of **15 predictors**, a starting model, hyperparameter tuning, probability calibration, and a missingness sensitivity analysis.

The target is **`LAPSE`**: 1 indicates a qualifying cancellation during the policy-year or non-renewal at its end; 0 indicates no qualifying lapse. Population exclusions and label definitions follow notebooks 03 and 04.

| Analysis stage | Policy-year starts | Purpose |
|---|---|---|
| Training | 2014–2020 | Fit preprocessing and model parameters |
| Rolling-origin folds | Validation years 2017–2020 | Select hyperparameters using earlier training years |
| Development evaluation (`test`) | 2021–2022 | Compare model versions and select the XGBoost pipeline |
| Final holdout (`valid`) | 2023–August 2024 | Evaluate the frozen pipeline once |

**Primary evaluation:** log loss, Brier score, and calibration. **Classification evaluation:** accuracy, balanced accuracy, precision, recall, and F1 at a prespecified 0.50 threshold. **Additional evaluation:** ROC-AUC, average precision (PR-AUC/AP), and top-decile capture. No class rebalancing is applied.

**Inputs:** `data/processed/features.parquet`, `docs/feature_plan.csv`, and the fixed cleaning rules in `src/features.py`. Insurance data and trained models are not bundled with this notebook. Numerical results and figures are generated by execution on the processed dataset.

## 2. Libraries and Configuration
The notebook runs from the project repository and uses the existing processed data. The software environment requires Python, XGBoost 2.0 or later, DuckDB, pandas, NumPy, scikit-learn, Matplotlib, PyArrow, and joblib.

In [2]:
from pathlib import Path
import sys
import json
import hashlib
import importlib.metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb
import joblib
import xgboost as xgb

from sklearn.model_selection import ParameterGrid
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    log_loss, brier_score_loss, roc_auc_score, average_precision_score,
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix
)
from IPython.display import display, Markdown

ModuleNotFoundError: No module named 'duckdb'

In [3]:
# Fixed settings used across the analysis.
SEED = 42
TARGET = 'LAPSE'
CLASSIFICATION_THRESHOLD = 0.50
TRAIN_ROWS = 1_000_000
TUNE_ROWS = 500_000
N_TREES = 200
N_THREADS = 4
N_CHUNKS = 28
CHECK_YEARS = [2017, 2018, 2019, 2020]

# The final holdout is reserved until model selection is complete.
RUN_FINAL_HOLDOUT = False
RUN_SENSITIVITY = True

BASE_PARAMS = dict(
    objective='binary:logistic', eval_metric='logloss', tree_method='hist',
    enable_categorical=True, n_estimators=N_TREES, learning_rate=0.05,
    max_depth=4, min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
    reg_lambda=5, scale_pos_weight=1, n_jobs=N_THREADS, random_state=SEED
)

In [4]:
# Locate the project and define its input and output folders.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/features.py').exists()), None)
assert ROOT is not None, 'The notebook must run inside the Flood Risk repository.'
sys.path.insert(0, str(ROOT))
from src.features import apply_fixed_rules

FEATURES_FILE = ROOT / 'data/processed/features.parquet'
SHARED_IDS_FILE = ROOT / 'data/processed/samples/lapse_train_ids.parquet'
REPORTS = ROOT / 'reports'
MODEL_DIR = ROOT / 'data/processed/models/lapse_xgb'
PRED_DIR = ROOT / 'data/processed/predictions'

for folder in [REPORTS, MODEL_DIR, PRED_DIR, SHARED_IDS_FILE.parent]:
    folder.mkdir(parents=True, exist_ok=True)
assert FEATURES_FILE.exists(), 'Required input missing: data/processed/features.parquet.'

AssertionError: Required input missing: data/processed/features.parquet.

In [5]:
# Use one plotting style and record the software versions.
plt.rcParams.update({
    'figure.figsize': (8, 4.5), 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False
})
versions = {name: importlib.metadata.version(name) for name in
            ['xgboost', 'duckdb', 'pandas', 'numpy', 'scikit-learn']}
display(pd.Series(versions, name='Version').to_frame())

PackageNotFoundError: No package metadata was found for xgboost

## 3. Selected Predictors
The model is restricted to 15 predictors from the approved feature plan. The shortlist covers pricing, coverage, tenure, flood exposure, and policy history. It is prespecified on substantive grounds rather than selected using development or final-holdout outcomes; it is not claimed to be the empirically optimal subset.

Identifiers, split labels, and the target are excluded. Native categorical support and native handling of permitted missingness preserve **15 actual model input columns**, without one-hot expansion or added missing-indicator columns. The same shortlist should be used when refitting other algorithms for a controlled comparison.

In [ ]:
# Define the compact feature set and the role of each predictor.
feature_data = [
    ('premium', 'Annual premium level'),
    ('premium_change_pct', 'Premium change from the previous term'),
    ('mandatory_purchase', 'Mortgage-related insurance requirement'),
    ('tenure_years', 'Time since the original policy purchase'),
    ('new_business', 'First-year policy status'),
    ('term_no', 'Position in the policy renewal history'),
    ('zone_group', 'Rated flood-zone category'),
    ('building_coverage', 'Amount of building coverage'),
    ('contents_coverage', 'Amount of contents coverage'),
    ('building_deductible', 'Building deductible category'),
    ('occupancy_group', 'Property occupancy category'),
    ('primary_residence', 'Primary-residence status'),
    ('floors', 'Number-of-floors category'),
    ('had_prior_claim', 'Previous claim history'),
    ('returned_after_lapse', 'Previous return after a coverage gap')
]
SELECTED_FEATURES = [name for name, role in feature_data]

In [ ]:
# Restrict the approved plan to the selected predictors.
approved = pd.read_csv(ROOT / 'docs/feature_plan.csv')
approved = approved[approved['use'].astype(str).str.lower().eq('true')]
assert set(SELECTED_FEATURES).issubset(set(approved.feature))
plan = approved.set_index('feature').loc[SELECTED_FEATURES].reset_index()

FEATS = plan['feature'].tolist()
NUMS = plan.loc[plan['kind'].eq('num'), 'feature'].tolist()
CATS = plan.loc[plan['kind'].eq('cat'), 'feature'].tolist()
FLAGS = plan.loc[plan['kind'].eq('flag'), 'feature'].tolist()
MISSING_PLAN = plan.set_index('feature')['missing_plan'].fillna('').to_dict()
RANDOM_COLS = [f for f in NUMS if 'NO missing flag' in MISSING_PLAN[f]]
assert len(FEATS) == 15 and len(set(FEATS)) == 15
assert not set(FEATS) & {TARGET, 'period_id', 'period_year', 'split'}

### Table 1. Selected Features
The table lists the 15 input variables, their approved type, and their substantive role. The feature count refers to the actual predictor matrix passed to XGBoost.

In [ ]:
# Display and save the feature specification.
feature_table = pd.DataFrame(feature_data, columns=['Feature', 'Role'])
feature_table['Type'] = feature_table.Feature.map(plan.set_index('feature')['kind'])
feature_table.index = np.arange(1, len(feature_table) + 1)
feature_table.index.name = 'Number'
display(feature_table)
feature_table.to_csv(REPORTS / 'lapse_xgb_selected_features.csv')

In [ ]:
# Open the processed dataset and select the eligible lapse population.
con = duckdb.connect()
con.execute("SET memory_limit='4GB'")
con.execute(f'SET threads={N_THREADS}')
con.from_parquet(str(FEATURES_FILE)).create_view('source_features')
con.execute('CREATE VIEW lapse_pop AS SELECT * FROM source_features WHERE in_pop_lapse')

SELECT_COLS = ', '.join(['period_id', 'period_year'] + FEATS + [TARGET])
columns = {row[0] for row in con.execute('DESCRIBE lapse_pop').fetchall()}
assert set(FEATS + [TARGET, 'period_id', 'period_year', 'split']).issubset(columns)

In [ ]:
# Summarize the training and development populations.
summary = con.sql(f"""
    SELECT split, MIN(period_year) AS first_year, MAX(period_year) AS last_year,
           COUNT(*) AS policy_years, AVG({TARGET}) AS lapse_rate
    FROM lapse_pop
    WHERE split IN ('train', 'test')
    GROUP BY split
    ORDER BY first_year
""").df()
display(summary.assign(lapse_rate=summary['lapse_rate'].map('{:.2%}'.format)))

In [ ]:
# Check the agreed time splits and binary target.
invalid_train = con.sql("""SELECT COUNT(*) FROM lapse_pop
    WHERE split='train' AND period_year NOT BETWEEN 2014 AND 2020""").fetchall()[0][0]
invalid_test = con.sql("""SELECT COUNT(*) FROM lapse_pop
    WHERE split='test' AND period_year NOT BETWEEN 2021 AND 2022""").fetchall()[0][0]
invalid_labels = con.sql(f"""SELECT COUNT(*) FROM lapse_pop
    WHERE split IN ('train','test')
      AND ({TARGET} IS NULL OR {TARGET} NOT IN (0,1))""").fetchall()[0][0]

assert invalid_train == invalid_test == invalid_labels == 0
print('Input schema, time splits, and target checks passed.')

### 3.1 Data Preparation Functions
Preprocessing is learned from training rows only. Suspect missingness in `tenure_years` is filled using stable training-donor draws. Missing premium changes and missing binary flags use native XGBoost missing-value handling, retaining permitted missingness information without introducing additional predictors. Other numeric values follow the approved fill rule; categories use native categorical support.

Stable donor filling mitigates an obvious missingness marker but does not establish the absence of leakage. ACS features and the other suspect tenure variable are outside this compact feature set.

In [ ]:
def load_rows(split, where="TRUE"):
    assert split in {"train", "test", "valid"}
    return con.sql(f"SELECT {SELECT_COLS} FROM lapse_pop WHERE split='{split}' AND {where}").df()

In [ ]:
def stable_draw(ids, key, pool_size):
    # IDs control reproducible draws only; they never enter the model.
    return np.fromiter((int.from_bytes(hashlib.sha256(
        f"{SEED}|{key}|{value}".encode()).digest()[:8], 'little') % pool_size
        for value in ids.astype(str)), dtype=np.int64, count=len(ids))

In [ ]:
def fit_preprocessor(frame, features=FEATS):
    clean = apply_fixed_rules(frame)
    state = {'features': list(features), 'medians': {}, 'levels': {}, 'pools': {}}
    for feature in features:
        if feature in CATS:
            observed = clean[feature].dropna().astype(str).unique().tolist()
            state['levels'][feature] = sorted(set(observed + ['__MISSING__']))
        else:
            values = pd.to_numeric(clean[feature], errors='coerce')
            values = values.replace([np.inf, -np.inf], np.nan)
            state['medians'][feature] = float(values.median()) if values.notna().any() else 0.0
            if feature in RANDOM_COLS:
                donors = values.dropna()
                assert len(donors), f'No training donors for {feature}.'
                pool = donors.sample(n=min(100_000, len(donors)), random_state=SEED)
                state['pools'][feature] = pool.to_numpy()
    return state

In [ ]:
def transform_features(frame, state):
    clean = apply_fixed_rules(frame).reset_index(drop=True)
    result = pd.DataFrame(index=clean.index)
    for feature in state['features']:
        if feature in CATS:
            values = clean[feature].astype('string').fillna('__MISSING__')
            result[feature] = pd.Categorical(values, categories=state['levels'][feature])
            continue
        values = pd.to_numeric(clean[feature], errors='coerce')
        values = values.replace([np.inf, -np.inf], np.nan)
        rule = MISSING_PLAN.get(feature, '')
        if feature in state['pools']:
            pool = state['pools'][feature]
            indexes = stable_draw(clean['period_id'], feature, len(pool))
            values = values.fillna(pd.Series(pool[indexes], index=clean.index))
        elif rule.startswith('fill 99'):
            values = values.fillna(99)
        elif feature in FLAGS or rule.startswith('fill 0') or '+ missing flag' in rule:
            pass  # Native NaN handling preserves permitted missingness within one column.
        else:
            values = values.fillna(state['medians'][feature])
        result[feature] = values.astype('float32')
    assert result.columns.tolist() == state['features']
    return result

## 4. Training Sample and Preprocessing
A fixed training-ID manifest preserves the sample across runs. An algorithm-only comparison with Random Forest requires the same training IDs and source-data version. Without a shared manifest, the sample is generated from the training years and recorded for reconciliation.

In [ ]:
# Reuse shared training IDs when a manifest is available.
if SHARED_IDS_FILE.exists():
    con.from_parquet(str(SHARED_IDS_FILE)).create_view('shared_ids')
    train = load_rows('train', 'period_id IN (SELECT period_id FROM shared_ids)')
    expected = con.sql('SELECT COUNT(*) FROM shared_ids').fetchall()[0][0]
    assert len(train) == expected, 'Training IDs do not match the input dataset.'
    sample_origin = 'Existing training-ID manifest'
else:
    train = con.sql(f"""SELECT * FROM
        (SELECT {SELECT_COLS} FROM lapse_pop WHERE split='train')
        USING SAMPLE reservoir({TRAIN_ROWS} ROWS) REPEATABLE ({SEED})""").df()
    train[['period_id']].to_parquet(SHARED_IDS_FILE, index=False)
    sample_origin = 'New manifest; matching Random Forest training IDs is not yet verified'

train = train.sort_values('period_id').reset_index(drop=True)
assert train['period_id'].is_unique and train[TARGET].nunique() == 2
sample_hash = hashlib.sha256(chr(10).join(train['period_id'].astype(str)).encode()).hexdigest()
print(f'Training sample: {len(train):,} policy-years; lapse rate: {train[TARGET].mean():.2%}.')
print(sample_origin)

In [ ]:
# Fit preprocessing only on the training sample.
prep = fit_preprocessor(train)
X_train = transform_features(train, prep)
y_train = train[TARGET].astype('int8')

print('Predictor matrix:', X_train.shape)
print('Target vector:', y_train.shape)
assert X_train.shape[1] == 15, 'The final predictor matrix must contain exactly 15 columns.'

In [ ]:
# Verify that imputed values do not depend on batch size or row order.
check = train.head(min(100, len(train)))
whole = transform_features(check, prep)
cut = len(check) // 2
parts = pd.concat([
    transform_features(check.iloc[:cut], prep),
    transform_features(check.iloc[cut:], prep)
], ignore_index=True)
reverse = transform_features(check.iloc[::-1], prep).iloc[::-1].reset_index(drop=True)

pd.testing.assert_frame_equal(whole, parts)
pd.testing.assert_frame_equal(whole, reverse)
print('Preprocessing reproducibility checks passed.')

## 5. Starting XGBoost Model
Version 1 uses fixed starting settings. The binary logistic objective estimates lapse probabilities. Class weights remain unadjusted to preserve the observed event scale.

In [ ]:
# Create the starting model and fit it to the training data.
starting_model = xgb.XGBClassifier(**BASE_PARAMS)
starting_model.fit(X_train, y_train)

print('Starting XGBoost model fitted.')

## 6. Hyperparameter Tuning
Four combinations of tree depth and minimum child weight are evaluated using expanding-year folds. Each fold fits its own preprocessing on earlier years and evaluates the following year. The selected combination minimizes mean fold log loss; development and final-holdout labels are excluded from tuning.

In [ ]:
def evaluate_candidate(parameters, frame):
    """Return time-fold scores and out-of-time predictions for one candidate."""
    fold_results, predictions = [], []
    for year in CHECK_YEARS:
        earlier = frame[frame.period_year < year]
        later = frame[frame.period_year == year]
        assert earlier[TARGET].nunique() == later[TARGET].nunique() == 2

        fold_prep = fit_preprocessor(earlier)
        fold_model = xgb.XGBClassifier(**{**BASE_PARAMS, **parameters})
        fold_model.fit(transform_features(earlier, fold_prep), earlier[TARGET])
        probabilities = fold_model.predict_proba(transform_features(later, fold_prep))[:, 1]

        fold_results.append({'year': year, 'log_loss': log_loss(
            later[TARGET], probabilities, labels=[0, 1])})
        predictions.append(pd.DataFrame({'y': later[TARGET].to_numpy(), 'p': probabilities}))
    return pd.DataFrame(fold_results), pd.concat(predictions, ignore_index=True)

In [ ]:
# Define a compact search grid and a fixed tuning sample.
parameter_grid = list(ParameterGrid({
    'max_depth': [3, 5],
    'min_child_weight': [20, 100]
}))
tune = train.sample(n=min(TUNE_ROWS, len(train)), random_state=SEED)
tune = tune.sort_values('period_id')

display(pd.DataFrame(parameter_grid))

In [ ]:
# Evaluate each candidate using the same time folds.
trial_predictions = {}
tuning_parts = []
for candidate_id, parameters in enumerate(parameter_grid):
    fold_scores, predictions = evaluate_candidate(parameters, tune)
    fold_scores = fold_scores.assign(candidate=candidate_id, **parameters)
    tuning_parts.append(fold_scores)
    trial_predictions[candidate_id] = predictions
    print(f'Candidate {candidate_id}: mean log loss = {fold_scores.log_loss.mean():.5f}')

tuning = pd.concat(tuning_parts, ignore_index=True)

In [ ]:
# Select the lowest-loss candidate and refit it on the training sample.
tuning_summary = tuning.groupby('candidate').agg(
    mean_log_loss=('log_loss', 'mean'), std_log_loss=('log_loss', 'std')
).join(pd.DataFrame(parameter_grid))
best_id = int(tuning_summary.mean_log_loss.idxmin())
best_params = parameter_grid[best_id]
best_oof = trial_predictions[best_id]

tuned_model = xgb.XGBClassifier(**{**BASE_PARAMS, **best_params})
tuned_model.fit(X_train, y_train)

display(tuning_summary.sort_values('mean_log_loss').round(5))
print('Selected parameters:', best_params)

## 7. Probability Calibration and Benchmarks
An isotonic calibrator is fitted using the selected candidate's out-of-time training-fold predictions. Both raw and calibrated probabilities are retained for development evaluation. The naive benchmarks use the full training population: a constant training lapse rate and a new-business × mandatory-purchase segment rate.

In [ ]:
# Learn the calibration mapping without using development or holdout labels.
calibrator = IsotonicRegression(
    out_of_bounds='clip', y_min=1e-5, y_max=1 - 1e-5
)
calibrator.fit(best_oof['p'], best_oof['y'])

print(f'Calibration fitted on {len(best_oof):,} out-of-time training predictions.')

In [ ]:
# Compute the same training-derived benchmarks used by the baseline notebooks.
base_rate = float(con.sql(
    f"SELECT AVG({TARGET}) FROM lapse_pop WHERE split='train'"
).fetchall()[0][0])
segments = con.sql(f"""
    SELECT CAST(COALESCE(tenure_years >= 0 AND tenure_years < 1, FALSE) AS INTEGER) AS new_business,
           CAST(COALESCE(mandatory_purchase, FALSE) AS INTEGER) AS mandatory_purchase,
           AVG({TARGET}) AS rate
    FROM lapse_pop WHERE split='train' GROUP BY ALL
""").df()
segment_map = {(int(r.new_business), int(r.mandatory_purchase)): r.rate
               for r in segments.itertuples()}

print(f'Training lapse rate: {base_rate:.2%}')

## 8. Development Predictions and Model Evaluation
Every eligible development policy-year is scored in memory-bounded chunks. Class 1 denotes lapse. Accuracy, balanced accuracy, precision, recall, and F1 use a fixed probability threshold of **0.50**; this threshold is not optimized on the development or final holdout.

High accuracy can result from predicting the majority non-lapse class. The training-rate benchmark therefore appears alongside lapse recall and F1. Log loss, Brier score, and calibration remain the primary measures for probability estimation and model selection. The training accuracy is in-sample performance on the training sample, not a generalization estimate.

In [ ]:
MODELS = {
    'Base rate': 'p_base', 'Segment rate': 'p_segment',
    'XGBoost V1': 'p_xgb_start', 'XGBoost V2': 'p_xgb_tuned',
    'XGBoost V2 calibrated': 'p_xgb_cal'
}

def score_split(split, only=None):
    """Score each policy-year once; optionally score only the selected XGBoost."""
    parts = []
    for chunk in range(N_CHUNKS):
        condition = f"hash(CAST(period_id AS VARCHAR) || '{SEED}') % {N_CHUNKS} = {chunk}"
        raw = load_rows(split, condition)
        if raw.empty:
            continue
        clean = apply_fixed_rules(raw)
        predictors = transform_features(raw, prep)
        scored = raw[['period_id', 'period_year', TARGET]].rename(columns={TARGET: 'y'}).copy()
        scored['p_base'] = base_rate
        scored['p_segment'] = [segment_map.get((int(a), int(b)), base_rate) for a, b in
            zip(clean.new_business.fillna(0), clean.mandatory_purchase.fillna(0))]
        if only is None or only == 'p_xgb_start':
            scored['p_xgb_start'] = starting_model.predict_proba(predictors)[:, 1]
        if only is None or only in {'p_xgb_tuned', 'p_xgb_cal'}:
            tuned = tuned_model.predict_proba(predictors)[:, 1]
            if only != 'p_xgb_cal':
                scored['p_xgb_tuned'] = tuned
            if only != 'p_xgb_tuned':
                scored['p_xgb_cal'] = calibrator.predict(tuned)
        scored['suspect_missing'] = clean[RANDOM_COLS].isna().any(axis=1) if RANDOM_COLS else False
        parts.append(scored)
    result = pd.concat(parts, ignore_index=True).sort_values('period_id').reset_index(drop=True)
    expected = con.sql(f"SELECT COUNT(*) FROM lapse_pop WHERE split='{split}'").fetchall()[0][0]
    assert result.period_id.is_unique and len(result) == expected
    return result

In [ ]:
def metrics_table(scored):
    """Calculate overall and yearly scores for the available probability columns."""
    rows = []
    years = sorted(scored.period_year.unique())
    overall_scope = f'{years[0]}-{years[-1]}'
    groups = [(overall_scope, scored)] + [(str(y), g) for y, g in scored.groupby('period_year')]
    for scope, part in groups:
        for name, column in MODELS.items():
            if column not in part:
                continue
            y = part.y.to_numpy()
            probabilities = np.clip(part[column].to_numpy(), 1e-7, 1 - 1e-7)
            predicted_class = (probabilities >= CLASSIFICATION_THRESHOLD).astype('int8')
            precision, recall, f1, _ = precision_recall_fscore_support(
                y, predicted_class, average='binary', zero_division=0)
            top = np.argsort(-probabilities, kind='stable')[:max(1, int(np.ceil(len(y) * 0.1)))]
            rows.append(dict(
                scope=scope, model=name, rows=len(y), actual_rate=y.mean(),
                predicted_rate=probabilities.mean(), accuracy=accuracy_score(y, predicted_class),
                balanced_accuracy=balanced_accuracy_score(y, predicted_class),
                precision=precision, recall=recall, f1=f1, threshold=CLASSIFICATION_THRESHOLD,
                log_loss=log_loss(y, probabilities, labels=[0, 1]),
                brier=brier_score_loss(y, probabilities),
                roc_auc=roc_auc_score(y, probabilities) if len(np.unique(y)) == 2 else np.nan,
                pr_auc=average_precision_score(y, probabilities) if y.sum() else np.nan,
                top_decile_capture=y[top].sum() / y.sum() if y.sum() else np.nan
            ))
    result = pd.DataFrame(rows)
    reference = result[result.model.eq('Base rate')].set_index('scope').log_loss
    result['log_loss_skill'] = 1 - result.log_loss / result.scope.map(reference)
    return result

In [ ]:
# Predict lapse probabilities for all development policy-years.
pred = score_split('test')
pred.to_parquet(PRED_DIR / 'lapse_xgb_test.parquet', index=False)

results = metrics_table(pred)
overall = results[results.scope.eq('2021-2022')].sort_values('log_loss')

print(f'Development policy-years scored: {len(pred):,}')
display(overall[['model', 'log_loss', 'brier', 'roc_auc', 'pr_auc', 'top_decile_capture']].round(5))

In [ ]:
# Examine year-to-year stability before selecting the XGBoost pipeline.
yearly = results[~results.scope.eq('2021-2022')]
display(yearly[['scope', 'model', 'log_loss', 'brier', 'actual_rate', 'predicted_rate']].round(5))

selected = overall[overall.model.str.startswith('XGBoost')].iloc[0]
selected_name = selected['model']
selected_column = MODELS[selected_name]
print('Development-selected pipeline:', selected_name)

### Table 2. Training and Development Accuracy
All rows use the same 0.50 decision threshold and the same development labels. Precision, recall, and F1 refer to the lapse class. Values are proportions: 0.85 corresponds to 85%. The constant training-rate benchmark shows the accuracy obtainable without distinguishing individual policy risks.

In [ ]:
# Compute training-sample probabilities for the same model versions.
train_probabilities = pd.DataFrame(index=train.index)
train_probabilities['p_base'] = base_rate
clean_train = apply_fixed_rules(train)
train_probabilities['p_segment'] = [segment_map.get((int(a), int(b)), base_rate)
    for a, b in zip(clean_train.new_business.fillna(0), clean_train.mandatory_purchase.fillna(0))]
train_probabilities['p_xgb_start'] = starting_model.predict_proba(X_train)[:, 1]
train_probabilities['p_xgb_tuned'] = tuned_model.predict_proba(X_train)[:, 1]
train_probabilities['p_xgb_cal'] = calibrator.predict(train_probabilities['p_xgb_tuned'])
train_accuracy = {name: accuracy_score(y_train, train_probabilities[column] >= CLASSIFICATION_THRESHOLD)
                  for name, column in MODELS.items()}

In [ ]:
# Display and save the requested accuracy comparison table.
accuracy_table = overall[['model', 'accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1']].copy()
accuracy_table.insert(1, 'training_accuracy', accuracy_table.model.map(train_accuracy))
accuracy_table = accuracy_table.rename(columns={'accuracy': 'development_accuracy'})
display(accuracy_table.round(4))
accuracy_table.to_csv(REPORTS / 'lapse_xgb_accuracy_table.csv', index=False)

### Confusion Matrix at the Fixed Threshold
The matrix reports the selected XGBoost pipeline's correct decisions, missed lapses, and false alarms on the development population. Rows are actual outcomes and columns are predicted outcomes.

In [ ]:
# Display the development confusion matrix for the selected pipeline.
predicted_class = (pred[selected_column] >= CLASSIFICATION_THRESHOLD).astype('int8')
counts = confusion_matrix(pred.y, predicted_class, labels=[0, 1])
confusion_table = pd.DataFrame(counts,
    index=['Actual non-lapse', 'Actual lapse'],
    columns=['Predicted non-lapse', 'Predicted lapse'])
display(confusion_table)
confusion_table.to_csv(REPORTS / 'lapse_xgb_confusion_table.csv')

### Table 3. Probability and Ranking Performance
Lower log loss and Brier score indicate better probabilistic performance. ROC-AUC and average precision assess ranking; top-decile capture measures concentration of actual lapses in the highest-scored 10%.

In [ ]:
# Keep probability metrics separate from threshold-based accuracy.
probability_table = overall[['model', 'log_loss', 'brier', 'roc_auc', 'pr_auc', 'top_decile_capture']]
display(probability_table.round(5))
probability_table.to_csv(REPORTS / 'lapse_xgb_probability_table.csv', index=False)

### 8.1 Logistic Regression and Random Forest Comparison
Existing aggregate reports are included when scoring-population counts and lapse rates agree. These are pipeline comparisons: training sample sizes, categorical encodings, and missing-value treatments may differ. Row-level ID and label reconciliation remains necessary for the consolidated comparison in notebook 08.

Older reports may use more than 15 predictors. Their probabilities are contextual pipeline references; a controlled compact-model comparison requires refitting each algorithm on the same 15-feature list. Accuracy is not reconstructed from aggregate probability metrics.

In [ ]:
# Add available baseline and Random Forest results for the same period.
comparison = overall[['model', 'rows', 'actual_rate', 'log_loss', 'brier']].copy()
for filename in ['lapse_baseline_logreg_test_metrics.csv', 'lapse_rf_test_metrics.csv']:
    path = REPORTS / filename
    if not path.exists():
        continue
    other = pd.read_csv(path)
    other = other[other.scope.astype(str).eq('2021-2022')]
    other = other[other.model.str.contains('logistic|RF', case=False, regex=True)]
    assert other.rows.eq(len(pred)).all(), 'Scoring populations differ.'
    assert np.allclose(other.actual_rate, pred.y.mean()), 'Observed lapse rates differ.'
    comparison = pd.concat([comparison, other[comparison.columns]], ignore_index=True)

comparison = comparison.sort_values('log_loss').reset_index(drop=True)
display(comparison.round(5))

## 9. Visual Evaluation
Calibration curves compare predicted and observed lapse rates while keeping equal scores in the same bin. Risk-decile and capture plots describe prioritization under a fixed contact capacity. Feature importance describes model reliance rather than causal effects.

In [ ]:
def decile_table(scored, column):
    table = scored.sort_values([column, 'period_id'], ascending=[False,True]).copy()
    table['risk_group'] = np.minimum(np.arange(len(table))*10//len(table)+1, 10)
    table = table.groupby('risk_group').agg(rows=('y','size'), lapses=('y','sum'),
        predicted=(column,'mean'), actual=('y','mean'), suspect_missing_share=('suspect_missing','mean'))
    table['share_of_lapses'] = table.lapses / table.lapses.sum()
    table['cumulative_capture'] = table.share_of_lapses.cumsum()
    table['lift'] = table.actual / scored.y.mean()
    return table

In [ ]:
# Summarize the selected pipeline by risk decile.
deciles = decile_table(pred, selected_column)
display(deciles[['rows', 'lapses', 'predicted', 'actual', 'share_of_lapses', 'lift']].round(4))

In [ ]:
# Plot the calibration of each XGBoost version.
fig, ax = plt.subplots()
for name in ['XGBoost V1', 'XGBoost V2', 'XGBoost V2 calibrated']:
    points = pd.DataFrame({'predicted': pred[MODELS[name]], 'actual': pred.y})
    points['bin'] = (pd.qcut(points.predicted, 10, duplicates='drop')
                     if points.predicted.nunique() > 1 else 0)
    grouped = points.groupby('bin', observed=True)[['predicted', 'actual']].mean()
    ax.plot(grouped.predicted, grouped.actual, marker='o', label=name)

limit = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, limit], [0, limit], '--', color='gray', label='Perfect calibration')
ax.set(title='Lapse calibration | Development years 2021–2022',
       xlabel='Mean predicted lapse probability', ylabel='Observed lapse rate')
ax.legend()
fig.tight_layout()
fig.savefig(REPORTS / 'fig_lapse_xgb_calibration.png', dpi=150)
plt.show()

In [ ]:
# Plot the observed lapse rate across risk deciles.
fig, ax = plt.subplots()
ax.bar(deciles.index, deciles.actual * 100, color='#167d9a')
ax.axhline(pred.y.mean() * 100, linestyle='--', color='gray', label='Overall lapse rate')
ax.set(title='Lapse rate by risk decile | Development years 2021–2022',
       xlabel='Risk decile (1 = highest risk)', ylabel='Observed lapse rate (%)')
ax.set_xticks(range(1, 11))
ax.legend()
fig.tight_layout()
fig.savefig(REPORTS / 'fig_lapse_xgb_deciles.png', dpi=150)
plt.show()

In [ ]:
# Plot the cumulative share of observed lapses captured.
fig, ax = plt.subplots()
coverage = np.arange(1, len(deciles) + 1) * 100 / len(deciles)
ax.plot(coverage, deciles.cumulative_capture * 100, marker='o', label=selected_name)
ax.plot([0, 100], [0, 100], '--', color='gray', label='Random selection')
ax.set(title='Cumulative lapse capture | Development years 2021–2022',
       xlabel='Policy-years targeted (%)', ylabel='Observed lapses captured (%)')
ax.legend()
fig.tight_layout()
fig.savefig(REPORTS / 'fig_lapse_xgb_capture.png', dpi=150)
plt.show()

In [ ]:
# Plot the features most strongly used by the selected XGBoost model.
chosen_model = starting_model if selected_column == 'p_xgb_start' else tuned_model
importance = pd.Series(chosen_model.feature_importances_, index=X_train.columns)
importance = importance.sort_values(ascending=False)
importance_table = importance.rename_axis('Feature').reset_index(name='Importance')
importance_table.index = np.arange(1, len(importance_table) + 1)
importance_table.index.name = 'Rank'
display(importance_table.round(5))

fig, ax = plt.subplots(figsize=(8, 6))
importance.head(15).sort_values().plot.barh(ax=ax, color='#167d9a')
ax.set(title=f'Feature importance | {selected_name}', xlabel='Normalized importance (gain)')
fig.tight_layout()
fig.savefig(REPORTS / 'fig_lapse_xgb_importance.png', dpi=150)
plt.show()

## 10. Missingness Diagnostics and Sensitivity Analysis
Only `tenure_years` among the selected predictors has a suspect missingness rule. Its original missingness status is retained outside the predictor matrix for diagnostics. A second experiment excludes this variable, producing a 14-predictor model with the same sample, tuned settings, and development population. This is a sensitivity check rather than an independently tuned competitor.

In [ ]:
# Compare original suspect-feature availability without using it as a predictor.
missingness = pred.groupby('suspect_missing').agg(
    policy_years=('y', 'size'), observed_rate=('y', 'mean'),
    predicted_rate=(selected_column, 'mean')
)
display(missingness.round(4))
print(f'Suspect-feature missingness overall: {pred.suspect_missing.mean():.1%}')
print(f'Missingness in the highest-risk decile: {deciles.suspect_missing_share.iloc[0]:.1%}')

In [ ]:
def sensitivity_experiment():
    """Evaluate the tuned configuration after excluding suspect features."""
    features = [f for f in FEATS if f not in RANDOM_COLS]
    state = fit_preprocessor(train, features=features)
    model = xgb.XGBClassifier(**{**BASE_PARAMS, **best_params})
    model.fit(transform_features(train, state), y_train)
    parts = []
    for chunk in range(N_CHUNKS):
        condition = f"hash(CAST(period_id AS VARCHAR) || '{SEED}') % {N_CHUNKS} = {chunk}"
        raw = load_rows('test', condition)
        if raw.empty:
            continue
        probabilities = model.predict_proba(transform_features(raw, state))[:, 1]
        parts.append(pd.DataFrame({'y': raw[TARGET].to_numpy(), 'p': probabilities}))
    scored = pd.concat(parts, ignore_index=True)
    return pd.DataFrame([dict(
        model='Tuned XGBoost with suspect tenure feature excluded', rows=len(scored),
        log_loss=log_loss(scored.y, scored.p, labels=[0, 1]),
        brier=brier_score_loss(scored.y, scored.p)
    )])

In [ ]:
# Record the sensitivity result without changing the selected feature list.
sensitivity = None
if RUN_SENSITIVITY:
    sensitivity = sensitivity_experiment()
    reference = overall[overall.model.eq('XGBoost V2')][['model', 'rows', 'log_loss', 'brier']]
    sensitivity_comparison = pd.concat([reference, sensitivity], ignore_index=True)
    display(sensitivity_comparison.round(5))
    sensitivity_comparison.to_csv(REPORTS / 'lapse_xgb_sensitivity.csv', index=False)

## 11. Final Holdout Evaluation
The selected pipeline is frozen using development results before final evaluation. `RUN_FINAL_HOLDOUT` defaults to `False`. When enabled, only the selected XGBoost version and the training-derived benchmarks are evaluated on the final population. Existing final predictions are not overwritten.

In [ ]:
# Evaluate the selected pipeline on the final holdout only when enabled.
final_results = None
if RUN_FINAL_HOLDOUT:
    holdout_path = PRED_DIR / 'lapse_xgb_valid.parquet'
    assert not holdout_path.exists(), 'Final predictions already exist; repeated holdout evaluation is blocked.'
    invalid_years = con.sql("""SELECT COUNT(*) FROM lapse_pop
        WHERE split='valid' AND period_year NOT BETWEEN 2023 AND 2024""").fetchall()[0][0]
    assert invalid_years == 0

    final_pred = score_split('valid', only=selected_column)
    final_results = metrics_table(final_pred)
    final_pred.to_parquet(holdout_path, index=False)
    final_results.to_csv(REPORTS / 'lapse_xgb_valid_metrics.csv', index=False)
    display(final_results[['scope', 'model', 'accuracy', 'precision', 'recall', 'f1', 'log_loss', 'brier']].round(5))
else:
    print('Final holdout status: not evaluated. Reported model comparisons are development results.')

## 12. Results Summary
The summary below is calculated directly from the executed model outputs. Development findings are distinguished from final-holdout evidence.

In [ ]:
# Generate the summary from the selected pipeline's actual evaluation scores.
reference_loss = float(overall.loc[overall.model.eq('Base rate'), 'log_loss'].iloc[0])
skill = 1 - float(selected['log_loss']) / reference_loss
capture = float(selected['top_decile_capture'])
calibration_helped = (overall.loc[overall.model.eq('XGBoost V2 calibrated'), 'log_loss'].iloc[0]
                     < overall.loc[overall.model.eq('XGBoost V2'), 'log_loss'].iloc[0])
status = ('Final holdout evaluated; scores are reported in Section 11.' if final_results is not None
          else 'The final holdout has not been evaluated; these findings are development results.')

display(Markdown(f"""**Development-selected pipeline:** {selected_name}.  
**Development period:** 2021–2022, comprising {len(pred):,} eligible policy-years.  
**Predictors:** {len(FEATS)} actual input columns.  
**Development accuracy:** {selected['accuracy']:.2%}; **lapse recall:** {selected['recall']:.2%}; **F1:** {selected['f1']:.4f} at threshold {CLASSIFICATION_THRESHOLD:.2f}.  
**Log loss:** {selected['log_loss']:.5f}; **Brier score:** {selected['brier']:.5f}.  
**Log-loss skill relative to the training-rate benchmark:** {skill:.1%}.  
**Highest-risk 10%:** captures {capture:.1%} of observed lapses.  
**Calibration:** {'improved' if calibration_helped else 'did not improve'} tuned-model development log loss.  
**Evaluation status:** {status}"""))

## 13. Saved Outputs and Reproducibility
Compact reports are saved under `reports/`. Models, preprocessing state, training IDs, and row-level predictions are saved under `data/processed/`. The selected prediction column, sample checksum, feature rules, and software versions identify the evaluated pipeline.

In [ ]:
# Save model files and tabular results.
starting_model.save_model(MODEL_DIR / 'xgb_start.json')
tuned_model.save_model(MODEL_DIR / 'xgb_tuned.json')
joblib.dump({'preprocessing': prep, 'calibrator': calibrator},
            MODEL_DIR / 'preprocessing.joblib')

results.to_csv(REPORTS / 'lapse_xgb_test_metrics.csv', index=False)
deciles.to_csv(REPORTS / 'lapse_xgb_test_deciles.csv')
tuning.to_csv(REPORTS / 'lapse_xgb_tuning.csv', index=False)
importance.rename('importance').to_csv(REPORTS / 'lapse_xgb_importance.csv')

In [ ]:
# Record the complete evaluation configuration.
run = dict(
    target=TARGET, seed=SEED, train_years='2014–2020', development_years='2021–2022',
    holdout_years='2023–August 2024', final_holdout_evaluated=final_results is not None,
    train_rows=len(train), tuning_rows=len(tune), sample_origin=sample_origin,
    sample_id_checksum=sample_hash, features=FEATS, suspect_features=RANDOM_COLS,
    feature_count=len(FEATS), model_columns=X_train.shape[1], classification_threshold=CLASSIFICATION_THRESHOLD,
    imputation='Stable training donors for tenure; permitted native missingness; no added predictors',
    class_weighting='None', parameters=BASE_PARAMS, tuned_parameters=best_params,
    selected_model=selected_name, selected_prediction_column=selected_column,
    calibration='Isotonic on out-of-time training-fold predictions',
    source_file=str(FEATURES_FILE), source_bytes=FEATURES_FILE.stat().st_size,
    source_mtime_ns=FEATURES_FILE.stat().st_mtime_ns,
    feature_plan_sha256=hashlib.sha256((ROOT / 'docs/feature_plan.csv').read_bytes()).hexdigest(),
    package_versions=versions, development_metrics=overall.to_dict(orient='records')
)
(REPORTS / 'lapse_xgb_run.json').write_text(json.dumps(run, indent=2), encoding='utf-8')

In [ ]:
# Confirm that the saved model reproduces the current predictions.
reloaded = xgb.XGBClassifier(enable_categorical=True)
reloaded.load_model(MODEL_DIR / 'xgb_tuned.json')
saved_state = joblib.load(MODEL_DIR / 'preprocessing.joblib')['preprocessing']
probe = train.head(100)

before = tuned_model.predict_proba(transform_features(probe, prep))[:, 1]
after = reloaded.predict_proba(transform_features(probe, saved_state))[:, 1]
np.testing.assert_allclose(before, after, rtol=1e-6)
print('Model files, reports, and prediction-reload checks completed.')

## 14. Limitations
- Reconstructed policy keys and redacted locations introduce linkage and measurement error.
- The target is dominated by non-renewal. Aggregate scores do not establish strong mid-term cancellation prediction.
- The 15-feature set is prespecified and has not been demonstrated to be the optimal subset. ACS and other excluded variables are not used in the compact model.
- Stable donor imputation mitigates an obvious tenure-missingness pattern but cannot certify the absence of leakage or preserve all relationships with other predictors.
- Calibration learned from training-fold predictions may transfer imperfectly to the refitted model and later periods.
- Benchmark comparisons involve complete pipelines. Sample sizes, encodings, preprocessing, source versions, and scoring IDs must be considered when attributing differences to algorithms.
- Accuracy depends on class balance and the decision threshold. High accuracy with low lapse recall is not evidence of strong lapse detection.
- Lapse risk identifies high-risk policy-years; it does not estimate the causal effect of retention outreach.

## References
1. [Project repository](https://github.com/jmsebastiancarino/Probabilistic-Model-for-Non-Life-Insurance-Flood-Risk): notebooks 03–06B, Random Forest notebook, `docs/feature_plan.csv`, and `docs/population_decisions.md`.
2. [XGBoost documentation](https://xgboost.readthedocs.io/en/stable/): binary classification, categorical support, and model persistence.
3. [scikit-learn probability calibration](https://scikit-learn.org/stable/modules/calibration.html): calibration curves and isotonic calibration.